# 📥 미래에셋증권 웹 수집 — 펀드 상세 페이지 SSR 필드

> 작성 2026-08-18 · 근거: `docs/DATA_COLLECTION_PLAN.md` (P0·P1) · `docs/DATA_NEEDS.md`
>
> **발견 (2026-08-18 엔드포인트 탐색)**
> - 상세 페이지 `GET /mw/mks/mks4116/p11.do?fd_cd=<itm_no>` — 마스터의 `itm_no`(KR 12자리)가 그대로 조회 키
> - 값은 ajax가 아니라 **SSR 인라인 JS 리터럴**로 내려옴: `ADMICN`(운용사 법인명) · `estd`(설정일) ·
>   `TFEE/SLE_FEE/ADMI_FEE/TRST_FEE/OFW_TRST_FEE`(보수 분해) · `NASST_SUM`(순자산) ·
>   `RPC_PHBT_YN`(환매금지) · `PD_CLSS/PD_TYP_CD/fdStcCd/spcDvCd`(사내 분류코드)
> - `POST /mw/mks/mks4116/a07.json` — **구성종목(최대 100행, ISIN·비중·기준일 `bas_dt`)**
> - 투자설명서류 URL은 크롤 없이 결정적으로 구성 가능:
>   `https://file.funddoctor.co.kr/app/file_download.asp?memb_cd=7070&file_gb=R{1,2,3,5}&pfund_cd=<itm_no>`
>   (R1 집합투자규약 · R2 투자설명서 · R3 간이투자설명서 · R5 운용보고서)
>
> **제약** — 페이지 값은 수집 시점(2026-08-18) 상태다. 불변 사실(설정일·법인명·구조)만 확정 근거로 쓰고,
> 시계열 값(순자산·수익률)은 **대조용으로만** 쓴다. Holdings는 `bas_dt ≤ 2026-07-11` 필터를 통과한 것만 채택.
> 마스터와 상충 시 마스터 우선 (`PROJECT.md` §3). 전 산출물에 `retrieved_at`·`source` 표기.

## 수집 세트 → 닫는 항목

| 세트 | 대상 | 닫는 항목 |
| :-- | :-- | :-- |
| `org` | 운용사 67코드 × 대표 종목(성공할 때까지 최대 3후보) | ① 운용사 법인명 (`ADMICN`) |
| `issu20`/`issu10_ctrl`/`issu00` | `fd_set_pcd` 값별 표본 25/15/10 | ⑬ 단위형 검증 · Q1 사각지대 실측 |
| `pfiv` | `pfiv_sale_cntl_tcd` `01`·`02` 클래스 전수 | ⑰ Q2 |
| `deriv06` | `or_attr_desc='06'` 표본 30 | ⑰ Q3 (페이지 `PD_TYP_CD` 대조) |
| `nav0` | `fd_nast_suma=0` & 판매중 표본 40 | ⑰ Q5 (페이지 `NASST_SUM` 대조) |
| `holdings` | 순자산 상위 10펀드 | ② Holdings 파일럿 (`a07.json`) |

In [1]:
# 셋업
import gzip, re, sqlite3, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "financial_products.db").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("repo root (data/financial_products.db) not found")
    ROOT = ROOT.parent

DB = ROOT / "data" / "financial_products.db"
OUT = ROOT / "data" / "external" / "miraeasset_web"
RAW = OUT / "raw"
RAW.mkdir(parents=True, exist_ok=True)

BASE = "https://securities.miraeasset.com"
DETAIL = BASE + "/mw/mks/mks4116/p11.do"
HOLDINGS_EP = BASE + "/mw/mks/mks4116/a07.json"
UA = ("Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) "
      "AppleWebKit/605.1.15 Version/17.0 Mobile/15E148 Safari/604.1")
SLEEP = 1.0                       # 요청 간격(예의)
AS_OF_LIMIT = "20260711"          # 데이터 기준일 (PROJECT.md §2-2)
RETRIEVED_AT = datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")
SRC_PAGE = "miraeasset_web(mks4116/p11.do)"
SRC_HOLD = "miraeasset_web(mks4116/a07.json)"

con = sqlite3.connect(DB)

def sql(q):
    return pd.read_sql(q, con)

print(ROOT)
print("retrieved_at =", RETRIEVED_AT)

C:\Users\NT751\Documents\toxin_2026\패류독소_작업\pj\mirae
retrieved_at = 2026-08-18T14:01:06+09:00


In [2]:
# 수집 대상 선정
targets = {}

# ① 운용사 코드별 대표 종목 후보 3개 (판매중 > 당사취급 우선)
targets["org"] = sql("""
with base as (
  select distinct or_co_xtn_itt_cd cd, itm_no, itm_abrv_nm, sale_yn,
         coalesce(thco_sale_yn,'') thco
  from public_funds
  where or_co_xtn_itt_cd is not null and itm_no like 'KR%'
), ranked as (
  select *, row_number() over (
      partition by cd
      order by (sale_yn='판매중') desc, (thco='Y') desc, itm_no) rn
  from base)
select * from ranked where rn <= 3
""")

# ⑬ fd_set_pcd 검증 세트
targets["issu20"] = sql("""
select distinct itm_no, itm_abrv_nm from public_funds
where fd_set_pcd='20' and sale_yn='판매중' and thco_sale_yn='Y'
order by itm_no limit 25""")
targets["issu10_ctrl"] = sql("""
select distinct itm_no, itm_abrv_nm from public_funds
where fd_set_pcd='10' and sale_yn='판매중' and thco_sale_yn='Y'
order by itm_no limit 15""")
targets["issu00"] = sql("""
select distinct itm_no, itm_abrv_nm from public_funds
where fd_set_pcd='00'
order by (coalesce(thco_sale_yn,'')='Y') desc, itm_no limit 10""")

# Q2 pfiv 01/02 전수
targets["pfiv"] = sql("""
select distinct itm_no, itm_abrv_nm, pfiv_sale_cntl_tcd from public_funds
where pfiv_sale_cntl_tcd in ('01','02')""")

# Q3 파생형 표본
targets["deriv06"] = sql("""
select distinct itm_no, itm_abrv_nm from public_funds
where or_attr_desc='06' and sale_yn='판매중' and thco_sale_yn='Y'
order by itm_no limit 30""")

# Q5 순자산 0 표본
targets["nav0"] = sql("""
select distinct itm_no, itm_abrv_nm from public_funds
where cast(fd_nast_suma as real)=0 and sale_yn='판매중' and thco_sale_yn='Y'
order by itm_no limit 40""")

for k, v in targets.items():
    print(f"{k:12s} {len(v):4d}")

flat_list = sorted(set().union(*[set(v.itm_no) for k, v in targets.items() if k != "org"]))
print("flat fetch list:", len(flat_list))

org           194
issu20         25
issu10_ctrl    15
issu00         10
pfiv           66
deriv06        30
nav0           40
flat fetch list: 185


In [3]:
# 수집·파싱 함수
S = requests.Session()
S.headers["User-Agent"] = UA

VAR_RE = re.compile(r'var\s+([A-Za-z_][A-Za-z0-9_]*)\s*=\s*"([^"\n]*)"\s*;')
SETVAL_RE = re.compile(r'\$\("#(\w+)"\)\.(?:html|text)\("([^"\n]*)"\)')
MOFUND_RE = re.compile(r'[가-힣A-Za-z0-9()\-&/·]+모투자신탁(?:\([^)<"]*\))?')

KEEP = ["FDN", "ADMICN", "estd", "TFEE", "SLE_FEE", "ADMI_FEE", "TRST_FEE",
        "OFW_TRST_FEE", "NASST_SUM", "ESTP", "M1_BNFR", "M3_BNFR", "M6_BNFR",
        "M12_BNFR", "RPC_PHBT_YN", "RPC_FEE_EXP", "RPC_TGM_GIV", "PD_CLSS",
        "PD_TYP_CD", "fdStcCd", "spcDvCd", "unibusYn", "HAN_CLAS_NM",
        "dms_fo_dv_cd", "bfFeeexp", "feeYn", "fdClssLv"]

def fetch_page(itm_no):
    """캐시 우선. 반환: (html or None, cached)."""
    raw = RAW / (itm_no + ".html.gz")
    if raw.exists():
        return gzip.decompress(raw.read_bytes()).decode("utf-8", "replace"), True
    r = S.get(DETAIL, params={"fd_cd": itm_no}, timeout=25)
    time.sleep(SLEEP)
    if r.status_code != 200:
        return None, False
    raw.write_bytes(gzip.compress(r.text.encode("utf-8")))
    return r.text, False

def parse_page(itm_no, html):
    d = {}
    for m in VAR_RE.finditer(html):
        k, v = m.group(1), m.group(2)
        if k in KEEP and v:
            d[k] = v          # 마지막 non-empty 우선
    for m in SETVAL_RE.finditer(html):
        k, v = m.group(1), m.group(2)
        if k in KEEP and v and k not in d:
            d[k] = v
    ok = bool(d.get("FDN"))
    row = {"itm_no": itm_no, "page_ok": ok,
           "mo_fund_names": ";".join(sorted(set(MOFUND_RE.findall(html)))) if ok else ""}
    row.update({k: d.get(k, "") for k in KEEP})
    return row

pages = {}

def collect(itm_no):
    if itm_no in pages:
        return pages[itm_no]
    try:
        html, _ = fetch_page(itm_no)
        row = parse_page(itm_no, html) if html else {"itm_no": itm_no, "page_ok": False}
    except Exception as e:
        row = {"itm_no": itm_no, "page_ok": False, "error": str(e)[:100]}
    pages[itm_no] = row
    return row

print("ready")

ready


In [4]:
# 수집 실행 — 운용사 세트(코드별 성공할 때까지) + 나머지 세트
org_rows = []
for cd, grp in targets["org"].groupby("cd"):
    hit_itm = None
    for _, r in grp.sort_values("rn").iterrows():
        if collect(r.itm_no).get("page_ok"):
            hit_itm = r.itm_no
            break
    org_rows.append({"or_co_xtn_itt_cd": cd, "itm_no": hit_itm})
print("org 코드 페이지 확보:", sum(1 for r in org_rows if r["itm_no"]), "/", len(org_rows))

for i, itm in enumerate(flat_list, 1):
    collect(itm)
    if i % 50 == 0:
        print(f"  {i}/{len(flat_list)}")

n_ok = sum(1 for p in pages.values() if p.get("page_ok"))
print(f"총 {len(pages)}페이지 요청 · 파싱 성공 {n_ok}")

org 코드 페이지 확보: 67 / 67


  50/185


  100/185


  150/185


총 249페이지 요청 · 파싱 성공 249


In [5]:
# 파싱 결과 적재 → fund_pages.csv
df = pd.DataFrame(pages.values())
df["retrieved_at"] = RETRIEVED_AT
df["source"] = SRC_PAGE
df["prospectus_url"] = ("https://file.funddoctor.co.kr/app/file_download.asp"
                        "?memb_cd=7070&file_gb=R2&pfund_cd=" + df["itm_no"])

master = sql("""
select itm_no, max(std_itm_no) std_itm_no, max(itm_abrv_nm) itm_abrv_nm,
       max(or_co_xtn_itt_cd) or_co_xtn_itt_cd, max(fd_set_pcd) fd_set_pcd,
       max(pfiv_sale_cntl_tcd) pfiv_sale_cntl_tcd, max(or_attr_desc) or_attr_desc,
       max(cast(fd_nast_suma as real)) fd_nast_suma,
       max(sale_yn) sale_yn, max(coalesce(thco_sale_yn,'')) thco_sale_yn
from public_funds group by itm_no""")
df = df.merge(master, on="itm_no", how="left")
df.to_csv(OUT / "fund_pages.csv", index=False, encoding="utf-8-sig")

print(df.groupby(["sale_yn", "thco_sale_yn"]).page_ok.agg(["sum", "count"]))

                      sum  count
sale_yn thco_sale_yn            
판매완료                   10     10
        Y              25     25
판매중     Y             214    214


In [6]:
# ① 운용사 코드 ↔ 법인명 (ADMICN) + 내부 브랜드 유도 대조 → org_name_map.csv
orgdf = pd.DataFrame(org_rows).merge(
    df[["itm_no", "ADMICN", "page_ok"]], on="itm_no", how="left")

names = sql("""select or_co_xtn_itt_cd cd, itm_abrv_nm
               from public_funds where or_co_xtn_itt_cd is not null""")
names["brand"] = names.itm_abrv_nm.str.extract(r"^([A-Za-z가-힣&]+)")
brand_mode = names.groupby("cd").brand.agg(
    lambda s: s.mode().iat[0] if not s.mode().empty else "")
n_itm = names.groupby("cd").size().rename("n_items")

orgdf = (orgdf.merge(brand_mode.rename("brand_rule"),
                     left_on="or_co_xtn_itt_cd", right_index=True, how="left")
              .merge(n_itm, left_on="or_co_xtn_itt_cd", right_index=True, how="left"))
orgdf["org_name"] = orgdf.ADMICN.fillna("")
orgdf["name_source"] = np.where(orgdf.org_name != "", SRC_PAGE, "rule(brand-prefix)")
orgdf.loc[orgdf.org_name == "", "org_name"] = orgdf.brand_rule
orgdf["retrieved_at"] = RETRIEVED_AT
orgdf = orgdf.sort_values("n_items", ascending=False)
orgdf.to_csv(OUT / "org_name_map.csv", index=False, encoding="utf-8-sig")

n_web = (orgdf.name_source == SRC_PAGE).sum()
print(f"법인명 웹 확보 {n_web}/67 · 브랜드 유도 대체 {67 - n_web}")
print(orgdf[["or_co_xtn_itt_cd", "org_name", "brand_rule", "name_source", "n_items"]]
      .head(20).to_string(index=False))
print("\n-- 웹 미확보 코드 --")
print(orgdf.loc[orgdf.name_source != SRC_PAGE,
                ["or_co_xtn_itt_cd", "brand_rule", "n_items"]].to_string(index=False))

법인명 웹 확보 67/67 · 브랜드 유도 대체 0
or_co_xtn_itt_cd      org_name      brand_rule                    name_source  n_items
        00080008      미래에셋자산운용   미래에셋전략배분적격TDF miraeasset_web(mks4116/p11.do)    24472
        00040010        삼성자산운용   삼성글로벌액티브적격TDF miraeasset_web(mks4116/p11.do)     8511
        00040035        KB자산운용      KB온국민적격TDF miraeasset_web(mks4116/p11.do)     6142
        00040024      한국투자신탁운용    한국투자적격TDF알아서 miraeasset_web(mks4116/p11.do)     5424
        00040067        신한자산운용     신한마음편한적격TDF miraeasset_web(mks4116/p11.do)     4168
        00040007        우리자산운용              우리 miraeasset_web(mks4116/p11.do)     3705
        00040027        한화자산운용 한화LIFEPLUS적격TDF miraeasset_web(mks4116/p11.do)     3649
        00080052      키움투자자산운용     키움키워드림적격TDF miraeasset_web(mks4116/p11.do)     3082
        00080029      피델리티자산운용       피델리티월드Big miraeasset_web(mks4116/p11.do)     2900
        00040040 NH-Amundi자산운용              NH miraeasset_web(mks4116/p11.do)     2778
        000800

In [7]:
# ⑬ fd_set_pcd 검증 — 페이지 사내코드·환매금지와 교차
sub = df[df.page_ok & df.fd_set_pcd.notna()]
for col in ["PD_CLSS", "fdStcCd", "spcDvCd", "RPC_PHBT_YN"]:
    print(f"=== fd_set_pcd × {col} ===")
    print(pd.crosstab(sub.fd_set_pcd, sub[col]))
    print()
# 단위형 신호: 종목명 토큰
sub20 = sub[sub.fd_set_pcd == "20"]
tok = sub20.itm_abrv_nm.str.contains("만기|목표전환", na=False)
print(f"'20' 표본 {len(sub20)}건 중 만기·목표전환 토큰 {tok.sum()}건")
print("'00' 페이지 확보:", int(df[df.fd_set_pcd == "00"].page_ok.sum()),
      "/", len(df[df.fd_set_pcd == "00"]), "(사각지대 실측)")

=== fd_set_pcd × PD_CLSS ===
PD_CLSS     00  11  12  13  14  15  16
fd_set_pcd                            
00           0   0   0   0   1   6   3
10           1  25  76  29  17  18  45
20           0   0   2   0   7  19   0

=== fd_set_pcd × fdStcCd ===
fdStcCd      00  02  03
fd_set_pcd             
00           10   0   0
10          176  14  21
20           28   0   0

=== fd_set_pcd × spcDvCd ===
spcDvCd      00  04  10  12  13
fd_set_pcd                     
00           10   0   0   0   0
10          184  11   8   5   3
20           21   3   0   0   4

=== fd_set_pcd × RPC_PHBT_YN ===
RPC_PHBT_YN    0  1
fd_set_pcd         
00             5  5
10           207  4
20            27  1

'20' 표본 28건 중 만기·목표전환 토큰 15건
'00' 페이지 확보: 10 / 10 (사각지대 실측)


In [8]:
# ⑰ Q2 — pfiv_sale_cntl_tcd 01 vs 02
q2 = df[df.pfiv_sale_cntl_tcd.isin(["01", "02"]) & df.page_ok]
print("페이지 확보:", len(q2), "/", int((df.pfiv_sale_cntl_tcd.isin(['01','02'])).sum()))
for col in ["HAN_CLAS_NM", "unibusYn", "spcDvCd", "PD_CLSS", "feeYn"]:
    print(f"=== pfiv × {col} ===")
    print(pd.crosstab(q2.pfiv_sale_cntl_tcd, q2[col]))
    print()

페이지 확보: 66 / 66
=== pfiv × HAN_CLAS_NM ===
HAN_CLAS_NM            수수료미징구-오프라인-고액  수수료미징구-오프라인-고액, 기관  수수료미징구-오프라인-고유재산  \
pfiv_sale_cntl_tcd                                                            
01                  6               0                   5                 1   
02                  0               1                   0                 0   

HAN_CLAS_NM         수수료미징구-오프라인-기관  수수료미징구-오프라인-기관, 고액  \
pfiv_sale_cntl_tcd                                       
01                              29                   1   
02                               9                   0   

HAN_CLAS_NM         수수료미징구-오프라인-기관, 법인100억  수수료미징구-오프라인-랩, 펀드 등  \
pfiv_sale_cntl_tcd                                                
01                                       1                    1   
02                                       0                    0   

HAN_CLAS_NM         수수료미징구-오프라인-변액보험특별계정  수수료미징구-오프라인-전문투자자  수수료미징구-오프라인-퇴직연금  \
pfiv_sale_cntl_tcd                                       

In [9]:
# ⑰ Q3 — or_attr_desc='06' 의 페이지 표기
q3 = df[(df.or_attr_desc == "06") & df.page_ok]
print("표본:", len(q3))
print("=== 페이지 PD_TYP_CD 분포 (마스터 '06' 대응 확인) ===")
print(q3.PD_TYP_CD.value_counts(dropna=False))
print("\n=== FDN(정식 펀드명)에 '파생' 포함 ===")
print(q3.FDN.str.contains("파생", na=False).value_counts())
print("\n예시 5건:")
print(q3[["itm_abrv_nm", "FDN", "PD_TYP_CD"]].head(5).to_string(index=False))

표본: 39
=== 페이지 PD_TYP_CD 분포 (마스터 '06' 대응 확인) ===
PD_TYP_CD
06    39
Name: count, dtype: int64

=== FDN(정식 펀드명)에 '파생' 포함 ===
FDN
True     38
False     1
Name: count, dtype: int64

예시 5건:
            itm_abrv_nm                                FDN PD_TYP_CD
한국투자미국MLP특별자(오일가스-파생)Ae 한국투자미국MLP특별자산자투자신탁(오일가스인프라-파생형)A-e        06
  삼성E스마트인덱스자투자제1호(주식파생)       삼성E-스마트인덱스증권자투자신탁제1호(주식-파생형)        06
  삼성중국레버리지자1(주식파생재간접)Ae  삼성중국본토레버리지증권자투자신탁제1호(주식-파생재간접형)Ae        06
  삼성중국레버리지자1(주식파생재간접)Ce  삼성중국본토레버리지증권자투자신탁제1호(주식-파생재간접형)Ce        06
  미래에셋재팬인덱스증권자1호(주-파)CP    미래에셋재팬인덱스증권자투자신탁1호(주식-파생형)종류C-P        06


In [10]:
# ⑰ Q5 — fd_nast_suma=0 vs 페이지 NASST_SUM
q5 = df[(df.fd_nast_suma == 0) & df.sale_yn.eq("판매중") & df.page_ok].copy()
q5["page_nav"] = pd.to_numeric(q5.NASST_SUM, errors="coerce")
print("표본:", len(q5))
print("페이지 순자산 > 0 :", int((q5.page_nav > 0).sum()))
print("페이지 순자산 = 0 :", int((q5.page_nav == 0).sum()))
print("파싱 불가:", int(q5.page_nav.isna().sum()))
print("\n예시 (마스터 0 · 페이지 non-zero):")
print(q5.loc[q5.page_nav > 0, ["itm_abrv_nm", "NASST_SUM", "estd"]]
      .head(8).to_string(index=False))
print("\n⚠️ 페이지값은 2026-08-18 시점 → 센티넬 '정황' 근거. 확정은 금투협 7/10 기준일 조회로.")

표본: 52
페이지 순자산 > 0 : 52
페이지 순자산 = 0 : 0
파싱 불가: 0

예시 (마스터 0 · 페이지 non-zero):
                  itm_abrv_nm  NASST_SUM     estd
        미래에셋아시아퍼시픽업종대표자1(주)C1    5411819 20070118
     미래에셋코친디아포커스7자1호(주식)(C-I) 1895612395 20070424
       미래에셋아시아퍼시픽인프라섹터1(주)C-i 9269200000 20070424
        미래에셋인디아인프라섹터자1호(주식)C1   16340319 20070713
       미래에셋아시아퍼시픽인프라섹터1(주)C-2 8385217052 20070720
        미래에셋차이나인프라섹터자1호(주식)C1   44131003 20070803
   미래에셋Eastern유라시아업종대표자1(주)C1        832 20070907
미래에셋EasternEURICS업종대표자1(주식)C1       8299 20070907

⚠️ 페이지값은 2026-08-18 시점 → 센티넬 '정황' 근거. 확정은 금투협 7/10 기준일 조회로.


In [11]:
# ② Holdings 파일럿 — a07.json (순자산 상위 10펀드)
hold_targets = sql("""
select itm_no, max(itm_abrv_nm) itm_abrv_nm, max(cast(fd_nast_suma as real)) nav
from public_funds
where sale_yn='판매중' and thco_sale_yn='Y'
group by itm_no order by nav desc limit 10""")

hrows = []
for _, r in hold_targets.iterrows():
    try:
        resp = S.post(HOLDINGS_EP,
                      data={"itm_no": r.itm_no, "all": "1", "stk_yn": "1",
                            "bd_yn": "1", "drvs_yn": "1", "liqt_yn": "1"},
                      headers={"X-Requested-With": "XMLHttpRequest",
                               "Referer": DETAIL + "?fd_cd=" + r.itm_no},
                      timeout=25)
        j = resp.json()
    except Exception as e:
        print(r.itm_no, "ERR", str(e)[:80])
        continue
    time.sleep(SLEEP)
    for g in j.get("grid01", []):
        hrows.append({"itm_no": r.itm_no, "fund_nm": r.itm_abrv_nm,
                      "bas_dt": j.get("bas_dt", ""),
                      "isin": g.get("zrin_itm_bztp_cd"),
                      "holding_nm": g.get("itm_bztp_nm"),
                      "weight_pct": g.get("fd_wtrt"),
                      "asset_type": g.get("ast_tp_nm"),
                      "market": g.get("mkt_tcd_nm")})

holds = pd.DataFrame(hrows)
if len(holds):
    holds["retrieved_at"] = RETRIEVED_AT
    holds["source"] = SRC_HOLD
    holds["as_of_ok"] = holds.bas_dt <= AS_OF_LIMIT   # 7/11 이전만 채택 가능
    holds.to_csv(OUT / "holdings_pilot.csv", index=False, encoding="utf-8-sig")
    print(holds.groupby(["fund_nm", "bas_dt", "as_of_ok"]).size()
          .rename("n_holdings").reset_index().to_string(index=False))
else:
    print("holdings 응답 없음")

                fund_nm   bas_dt  as_of_ok  n_holdings
IBK그랑프리국공채MMF법인1호[국공채]C 20260601      True          20
IBK그랑프리국공채MMF법인1호[국공채]I 20260601      True          20
       KB법인용 MMF I-1호CF 20260601      True          20
         KB법인용MMF I-1호C 20260601      True          20
           삼성MMF법인제1호 C 20260601      True          20
          삼성MMF법인제1호 Ce 20260601      True          20
          삼성MMF법인제1호 Cp 20260601      True          20
         삼성MMF법인제1호 Cpe 20260601      True          20
    우리큰만족법인MMF1(국공채)C-e 20260601      True          20
    우리큰만족법인MMF1호(국공채) C 20260601      True          20


In [12]:
# 결론 요약 + 산출물 README
n_req = len(df)
n_ok = int(df.page_ok.sum())
n_web_org = int((orgdf.name_source == SRC_PAGE).sum())
n_mo = int((df.page_ok & df.mo_fund_names.ne("")).sum())

summary = f"""# 미래에셋증권 웹 수집 산출물

수집일 {RETRIEVED_AT} · 노트북 `notebooks/collect_miraeasset_web.ipynb`

| 파일 | 내용 |
| :-- | :-- |
| `fund_pages.csv` | 상세 페이지 SSR 필드 {n_req}건 요청 · {n_ok}건 파싱 성공 |
| `org_name_map.csv` | 운용사 67코드 ↔ 법인명 — 웹 확보 {n_web_org} · 나머지 브랜드 유도(rule) |
| `holdings_pilot.csv` | a07.json 구성종목 파일럿 (bas_dt 포함, as_of_ok 필터 필수) |
| `raw/*.html.gz` | 페이지 원본 (재파싱용 캐시) |

⚠️ 페이지 값은 수집 시점 상태. 불변 사실(설정일 estd·법인명 ADMICN·모펀드명)만 확정 근거,
시계열(NASST_SUM·수익률)은 대조용. 마스터 상충 시 마스터 우선. as_of ≤ {AS_OF_LIMIT} 필터.
모펀드명 추출 {n_mo}건 (mo_fund_names — ③ 모자관계 소재).
"""
(OUT / "README.md").write_text(summary, encoding="utf-8")
print(summary)

# 미래에셋증권 웹 수집 산출물

수집일 2026-08-18T14:01:06+09:00 · 노트북 `notebooks/collect_miraeasset_web.ipynb`

| 파일 | 내용 |
| :-- | :-- |
| `fund_pages.csv` | 상세 페이지 SSR 필드 249건 요청 · 249건 파싱 성공 |
| `org_name_map.csv` | 운용사 67코드 ↔ 법인명 — 웹 확보 67 · 나머지 브랜드 유도(rule) |
| `holdings_pilot.csv` | a07.json 구성종목 파일럿 (bas_dt 포함, as_of_ok 필터 필수) |
| `raw/*.html.gz` | 페이지 원본 (재파싱용 캐시) |

⚠️ 페이지 값은 수집 시점 상태. 불변 사실(설정일 estd·법인명 ADMICN·모펀드명)만 확정 근거,
시계열(NASST_SUM·수익률)은 대조용. 마스터 상충 시 마스터 우선. as_of ≤ 20260711 필터.
모펀드명 추출 87건 (mo_fund_names — ③ 모자관계 소재).

